# PAUL Open Model - DPO V2 Training (Kaggle)

This notebook executes the DPO V2 corrective experiment on Kaggle.
It strictly validates the private SFT adapter and captures full provenance before launching training.

## 1. Repository Setup & Cleanliness
Clone the repository and explicitly checkout the canonical commit.

In [ ]:
import os
import sys
import subprocess

EXPECTED_COMMIT = "4dd51ad2cf150f8c5204c41d317d4a7f940a2a54"
REPO_URL = "https://github.com/paul-foundry/paul-open.git"
REPO_DIR = "/kaggle/working/paul-open"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "checkout", EXPECTED_COMMIT], check=True)

current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("utf-8").strip()
if current_commit != EXPECTED_COMMIT:
    print(f"CRITICAL ERROR: Git commit mismatch! Expected {EXPECTED_COMMIT}, got {current_commit}")
    sys.exit(1)

git_status = subprocess.check_output(["git", "status", "--short"]).decode("utf-8").strip()
if git_status:
    print(f"CRITICAL ERROR: Working tree is dirty!\n{git_status}")
    sys.exit(1)

print(f"Repository successfully locked to commit: {current_commit}")
!pip install -r requirements.txt

## 2. Pre-flight Provenance and Validation
Validate DPOTrainer semantics, exact adapter hashes, and generate pre-flight manifest.

In [ ]:
import hashlib
import json
import platform
import torch
import transformers
import trl
import peft
import bitsandbytes

def hash_file(path):
    h = hashlib.sha256()
    if not os.path.exists(path):
        return None
    with open(path, 'rb') as f:
        while chunk := f.read(8192):
            h.update(chunk)
    return h.hexdigest()

def hash_dir(dir_path):
    h = hashlib.sha256()
    for root, _, files in sorted(os.walk(dir_path)):
        for file in sorted(files):
            file_path = os.path.join(root, file)
            h.update(file.encode())
            with open(file_path, 'rb') as f:
                while chunk := f.read(8192):
                    h.update(chunk)
    return h.hexdigest()

# Validate TRL version for DPOTrainer API compatibility
TRL_MIN_VER = "0.10.0"
if trl.__version__ < TRL_MIN_VER:
    print(f"CRITICAL ERROR: TRL version {trl.__version__} is too old. DPOTrainer semantics require >= {TRL_MIN_VER}")
    sys.exit(1)
print(f"TRL version {trl.__version__} validated for DPOTrainer semantics.")

ADAPTER_DIR = "/kaggle/input/paul-open-sft-adapter"
adapter_config_path = os.path.join(ADAPTER_DIR, "adapter_config.json")
safetensors_path = os.path.join(ADAPTER_DIR, "adapter_model.safetensors")

print("\n--- VALIDATING ADAPTER ---")
config_hash = hash_file(adapter_config_path)
safetensors_hash = hash_file(safetensors_path)
dir_hash = hash_dir(ADAPTER_DIR)

if not config_hash or not safetensors_hash:
    print("CRITICAL ERROR: Adapter artifacts missing from Kaggle dataset.")
    sys.exit(1)

with open(adapter_config_path, 'r') as f:
    adapter_config = json.load(f)

adapter_base_model = adapter_config.get('base_model_name_or_path')
print(f"Adapter Config SHA-256: {config_hash}")
print(f"Adapter Model SHA-256: {safetensors_hash}")
print(f"Adapter Dir SHA-256: {dir_hash}")
print(f"Adapter Base Model: {adapter_base_model}")

data_hash = hash_file("data/train/dpo_v2_corrective.jsonl")
config_file_hash = hash_file("configs/training/dpo.yaml")
script_hash = hash_file("scripts/train.py")

env_info = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "transformers": transformers.__version__,
    "trl": trl.__version__,
    "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
}

preflight_manifest = {
    "experiment_id": "dpo_v2_corrective",
    "canonical_git_sha": EXPECTED_COMMIT,
    "verified_git_sha": current_commit,
    "dataset_hash": data_hash,
    "config_hash": config_file_hash,
    "train_script_hash": script_hash,
    "base_model": "configs/models/gemma4_12b_it.yaml",
    "adapter_hashes": {
        "config": config_hash,
        "safetensors": safetensors_hash,
        "directory": dir_hash
    },
    "adapter_provenance": adapter_base_model,
    "environment": env_info,
    "seed": 42
}

with open('dpo_v2_manifest_preflight.json', 'w') as f:
    json.dump(preflight_manifest, f, indent=2)
print("\nPre-flight manifest saved to dpo_v2_manifest_preflight.json")

## 3. Dry Run
Verify model loading, config parsing, and DPOTrainer construction.

In [ ]:
!python scripts/train.py --model configs/models/gemma4_12b_it.yaml \
                         --training configs/training/dpo.yaml \
                         --data data/train/dpo_v2_corrective.jsonl \
                         --adapter /kaggle/input/paul-open-sft-adapter \
                         --dry-run

## 4. Smoke Test
Force max_steps=1 to guarantee a forward pass, backward pass, and optimizer step safely without running full training.

In [ ]:
!python scripts/train.py --model configs/models/gemma4_12b_it.yaml \
                         --training configs/training/dpo.yaml \
                         --data data/train/dpo_v2_corrective.jsonl \
                         --adapter /kaggle/input/paul-open-sft-adapter \
                         --smoke-test

## 5. Execute Final Training
If dry run and smoke test pass, launch the complete DPO training.

In [ ]:
!python scripts/train.py --model configs/models/gemma4_12b_it.yaml \
                         --training configs/training/dpo.yaml \
                         --data data/train/dpo_v2_corrective.jsonl \
                         --adapter /kaggle/input/paul-open-sft-adapter

## 6. Generate Final Manifest
Package training results, output hashes, and final status.

In [ ]:
import os
import json

with open('dpo_v2_manifest_preflight.json', 'r') as f:
    manifest = json.load(f)

output_dir = "./results/dpo_v2"
output_safetensors = os.path.join(output_dir, "adapter_model.safetensors")

manifest["training_status"] = "COMPLETED" if os.path.exists(output_safetensors) else "FAILED"
manifest["actual_output_path"] = output_dir

if manifest["training_status"] == "COMPLETED":
    manifest["final_output_hashes"] = {
        "config": hash_file(os.path.join(output_dir, "adapter_config.json")),
        "safetensors": hash_file(output_safetensors),
        "directory": hash_dir(output_dir)
    }
else:
    manifest["final_output_hashes"] = None

with open('dpo_v2_manifest_final.json', 'w') as f:
    json.dump(manifest, f, indent=2)
print("Final manifest saved to dpo_v2_manifest_final.json")